# LoRA vs QLoRA

## Core Difference

The core difference between LoRA and QLoRA comes down to how the frozen base model is represented in memory during fine-tuning.

## Key Distinction

* **LoRA:** Fine-tunes a 16-bit base model with 16-bit adapters
* **QLoRA:** Compresses the frozen base model to 4-bit NF4 to drastically cut VRAM while keeping the trainable adapters in 16-bit

## QLoRA & Memory Bottleneck Reduction

QLoRA (Quantized Low-Rank Adaptation) reduces fine-tuning VRAM requirements by quantizing the frozen base model down to a specialized 4-bit representation while keeping the trainable LoRA adapter parameters in 16-bit floating-point precision ($BF16$ or $FP16$).

---

### 4-bit Quantization Mechanics & NormalFloat4 (NF4)

Standard integer quantization (like $INT4$) maps numbers to an evenly spaced linear grid. However, neural network weight parameters following pre-training are not uniformly distributed—they follow a zero-mean normal (Gaussian) distribution:

$$W \sim \mathcal{N}(0, \sigma^2)$$

Using uniform linear quantization grids on normally distributed weights causes massive information loss near the tails and wastes quantization levels in sparse regions.

**Linear INT4 Grid (Uniform Spacing):**

```
├───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┤
-7  -6  -5  -4  -3  -2  -1   0   1   2   3   4   5   6   7
```

**NormalFloat4 NF4 Grid (Information-Theoretically Spaced):**

```
├─┼──┼───┼────┼─────┼──────┼───────┼───────┼──────┼─────┼────┼───┼──┼─┤
Dense levels near zero (high weight concentration), sparse at tails
```

---

### NormalFloat4 (NF4) Construction

NormalFloat4 (NF4) is an information-theoretically optimal quantile quantization data type for normally distributed data.

Instead of setting bin boundaries at fixed intervals, NF4 constructs $2^k = 16$ quantization bins $q_i$ ($i = 0, \dots, 15$) such that each bin has an equal probability of containing a weight value:

$$q_i = \frac{1}{2} \left( Q_X\left(\frac{i}{2^k}\right) + Q_X\left(\frac{i+1}{2^k}\right)\right)$$

Where $Q_X(\cdot)$ is the quantile function (inverse CDF) of a standard normal distribution $\mathcal{N}(0, 1)$ normalized to the range $[-1, 1]$.

**The 16 Discrete NF4 Values**

The exact hardcoded 4-bit lookup values for NF4 are:

$$[-1.0, -0.6961, -0.5251, -0.3949, -0.2844, -0.1848, -0.0910, 0.0, 0.0796, 0.1609, 0.2471, 0.3449, 0.4631, 0.6106, 0.8415, 1.0]$$

Notice how $0.0$ is explicitly represented to ensure zero-padding and sparse operations remain exact, while values cluster densely around $0.0$.

---

### Block-Wise Quantization & Double Quantization (DQ)

Quantizing a weight tensor $W$ requires scaling it to the $[-1, 1]$ NF4 range using scale factors $c$.

**Standard Block-Wise Quantization**

To prevent extreme outliers in one part of a large weight matrix from ruining precision elsewhere, $W$ is chunked into small independent blocks (typically block size $B = 64$ or $B = 128$ parameters).

For each block of 64 parameters:

1. Find absolute max: $c_1 = \max(\vert{}W\vert{})$
2. Quantize block weights to 4-bit NF4 indices: $q_i = \text{Quantize}(W_i / c_1)$

**VRAM Footprint of Scales:**

* If block size $B = 64$, every 64 parameters (which take $64 \times 0.5 \text{ bytes} = 32 \text{ bytes}$ in 4-bit) require one 32-bit float scale factor $c_1$ ($4 \text{ bytes}$)
* Memory overhead of scale factors = $\frac{32 \text{ bits}}{64 \text{ parameters}} = 0.5 \text{ bits/param}$
* Effective base model footprint = $4.0 + 0.5 = 4.5 \text{ bits/param}$

**Double Quantization (DQ) Mechanics**

Double Quantization quantizes the quantization constants $c_1$ themselves!

1. Group the 32-bit FP32 scale factors $c_1$ into secondary blocks of size $B_2 = 256$
2. Compute a second FP32 scale factor $c_2 = \max(\vert{}c_1\vert{})$
3. Quantize the primary scale factors $c_1$ from FP32 (32 bits) down to FP8 (8 bits)

**Memory Footprint Reduction with DQ:**

$$\text{FP8 Quantized Scale Overhead} = \frac{8 \text{ bits}}{64 \text{ params}} = 0.125 \text{ bits/param}$$

$$\text{Secondary FP32 Scale Overhead} = \frac{32 \text{ bits}}{64 \times 256 \text{ params}} = 0.00195 \text{ bits/param}$$

$$\text{Total Effective Footprint with DQ} = 4.0 + 0.125 + 0.00195 \approx \mathbf{4.127 \text{ bits/param}}$$

By applying Double Quantization, we save $\sim 0.373$ bits per parameter, which equals $\sim 320 \text{ MB}$ of VRAM saved per 7 Billion parameters!

## The Core Mechanical Difference

### LoRA (16-bit Base + 16-bit Adapters)

**Base Model Weights ($W_0$):**
* Frozen in 16-bit precision (BF16 or FP16)
* For a 7B model, base weights alone take $\sim 14\text{ GB}$ of VRAM

**Adapters ($A$ and $B$):**
* Trained in 16-bit precision (BF16 or FP16)

**Forward Math:**
* Standard matrix multiplication in 16-bit: $y = x \cdot W_0 + \frac{\alpha}{r} (x \cdot B \cdot A)$

---

### QLoRA (4-bit Quantized Base + 16-bit Adapters)

**Base Model Weights ($W_0$):**
* Quantized and frozen in 4-bit precision using NormalFloat4 (NF4) + Double Quantization (DQ)
* For a 7B model, base weights shrink from $\sim 14\text{ GB}$ down to $\sim 3.8\text{ GB}$ of VRAM

**Adapters ($A$ and $B$):**
* Trained in 16-bit precision (BF16/FP16)

**Forward Math:**
* As the input tensor $x$ hits a layer:
  1. The 4-bit base weights are dequantized on-the-fly into 16-bit registers in GPU cache
  2. Multiplied by $x$
  3. Added to the 16-bit LoRA adapter result
  4. The 16-bit base weights are discarded (not stored in activations)